# Итеративная Линейная регрессия

In [ ]:
class GDLinearRegression:
    def __init__(self, learning_rate=0.01, tolerance=1e-12, max_iterations=1000):
        self.learning_rate = learning_rate
        self.tolerance = tolerance
        self.max_iterations = max_iterations

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.bias = 0.0
        self.weights = np.zeros(n_features)
        
        loss_history = []
        
        for iteration in range(self.max_iterations):
            y_pred = X @ self.weights + self.bias
            loss = np.mean((y_pred - y) ** 2)
            loss_history.append(loss)
            
            db = 2 / n_samples * np.sum(y_pred - y)  
            dw = 2 / n_samples * X.T @ (y_pred - y)
            
            self.bias -= self.learning_rate * db    #Просто движение к минимуму слева требует увеличения w, а формула с минусом это автоматически обеспечивает.
            self.weights -= self.learning_rate * dw  #АНТИГРАДИЕНТ
            
            self.weights = np.maximum(self.weights, 0)
            
            # Проверка сходимости по величине градиента (знаечния уже почти не меняются)
            if np.abs(db) < self.tolerance and np.all(np.abs(dw) < self.tolerance):
                print(f"Сошелся за {iteration} итераций, финальный Loss: {loss:.6f}")
                break
        
        return loss_history

    def predict(self, X_test):
        return X_test @ self.weights + self.bias

# Знакомство с nnls

In [ ]:
from scipy.optimize import nnls
from sklearn.linear_model import LinearRegression
import time

start = time.time()
x_nnls, resid = nnls(A, b)
time_nnls = time.time() - start

ols = LinearRegression(fit_intercept=False, positive=True)
start = time.time()
ols.fit(A, b)
x_ols = ols.coef_
time_ols = time.time() - start

x_nnls_norm = x_nnls / x_nnls.sum()
x_ols_norm = x_ols / x_ols.sum() if x_ols.sum() != 0 else x_ols

print("=" * 50)
print("РЕЗУЛЬТАТЫ НА СЛОЖНЫХ ДАННЫХ")
print("=" * 50)
print(f"Истинные:        {true_proportions}")
print(f"NNLS scipy:            {x_nnls_norm}")
print(f"NNLS scikit:             {x_ols_norm}")
print("-" * 50)
print(f"Ошибка NNLS scipy:     {np.linalg.norm(x_nnls_norm - true_proportions):.4f}")
print(f"Ошибка NNLS scikit:      {np.linalg.norm(x_ols_norm - true_proportions):.4f}")
print(f"Время NNLS scipy:      {time_nnls:.4f} сек")
print(f"Время NNLS scikit:       {time_ols:.4f} сек")

# Преобразование признаков и линейность по параметрам (сгенерировано, просто для фактчекинга)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Создаём данные
data = {
    'Опыт (годы)': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 13, 13, 14, 15],
    'Зарплата (тыс. руб)': [30, 35, 60, 80, 105, 145, 170, 190, 205, 215, 220, 220, 222, 221, 223, 222, 230]
}
df = pd.DataFrame(data)

# Признак и целевая переменная
X_original = df[['Опыт (годы)']].values
X_log = np.log(df[['Опыт (годы)']]).values  # логарифмическое преобразование
y = df['Зарплата (тыс. руб)'].values

# --- 1. Обычная линейная регрессия ---
model_lin = LinearRegression()
model_lin.fit(X_original, y)
y_pred_lin = model_lin.predict(X_original)

# --- 2. Линейная регрессия с логарифмическим признаком ---
model_log = LinearRegression()
model_log.fit(X_log, y)
y_pred_log = model_log.predict(X_log)

# --- 3. Метрики качества ---
print("=" * 60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 60)

print("\n=== Обычная линейная регрессия ===")
print(f"Уравнение: Зарплата = {model_lin.intercept_:.2f} + {model_lin.coef_[0]:.2f} * Опыт")
print(f"R² = {r2_score(y, y_pred_lin):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, y_pred_lin)):.2f} тыс. руб")

print("\n=== Линейная регрессия с ln(Опыт) ===")
print(f"Уравнение: Зарплата = {model_log.intercept_:.2f} + {model_log.coef_[0]:.2f} * ln(Опыт)")
print(f"R² = {r2_score(y, y_pred_log):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y, y_pred_log)):.2f} тыс. руб")

# --- 4. Сравнение предсказаний для новых данных ---
print("\n" + "=" * 60)
print("ПРОГНОЗ ДЛЯ НОВЫХ ДАННЫХ")
print("=" * 60)

test_exp = np.array([[15], [16],[17],[18],[19], [20], [25]])
print("\nОбычная модель:")
for exp in test_exp:
    pred = model_lin.predict(exp.reshape(-1, 1))[0]
    print(f"  {exp[0]} лет опыта → {pred:.0f} тыс. руб")

print("\nЛогарифмическая модель:")
for exp in test_exp:
    pred = model_log.predict(np.log(exp.reshape(-1, 1)))[0]
    print(f"  {exp[0]} лет опыта → {pred:.0f} тыс. руб")

print("\n(По логике, зарплата не должна сильно превышать 220-230 тыс. руб)")

# --- 5. Визуализация ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График 1: Обычная регрессия
axes[0].scatter(X_original, y, color='blue', s=50, label='Фактические данные')
axes[0].plot(X_original, y_pred_lin, color='red', linewidth=2, label='Линейная регрессия')
axes[0].set_title('Обычная линейная регрессия', fontsize=14)
axes[0].set_xlabel('Опыт (годы)', fontsize=12)
axes[0].set_ylabel('Зарплата (тыс. руб)', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)
# Добавляем аннотацию с ошибкой на последней точке
axes[0].annotate(f'Ошибка: {abs(y[-1] - y_pred_lin[-1]):.1f} тыс.',
                 xy=(13, y[-1]), xytext=(10.5, 190),
                 arrowprops=dict(arrowstyle='->', color='gray'))

# График 2: Регрессия с логарифмом (В исходном пространстве)
axes[1].scatter(X_original, y, color='blue', s=50, label='Фактические данные')
# Для отображения кривой нужно отсортировать X
X_sorted = np.sort(X_original, axis=0)
X_log_sorted = np.log(X_sorted)
y_pred_log_sorted = model_log.predict(X_log_sorted)
axes[1].plot(X_sorted, y_pred_log_sorted, color='green', linewidth=2, label='Регрессия с ln(Опыт)')
axes[1].set_title('Линейная регрессия с логарифмическим признаком', fontsize=14)
axes[1].set_xlabel('Опыт (годы)', fontsize=12)
axes[1].set_ylabel('Зарплата (тыс. руб)', fontsize=12)
axes[1].legend()
axes[1].grid(True, alpha=0.3)
# Добавляем аннотацию с ошибкой на последней точке
axes[1].annotate(f'Ошибка: {abs(y[-1] - y_pred_log[-1]):.1f} тыс.',
                 xy=(13, y[-1]), xytext=(10.5, 190),
                 arrowprops=dict(arrowstyle='->', color='gray'))

plt.tight_layout()
plt.show()

# --- 6. Дополнительный график: линеаризация (в обновленном пространстве с логарифмом на оси ---
plt.figure(figsize=(8, 5))
plt.scatter(np.log(X_original), y, color='purple', s=50)
plt.title('После преобразования: ln(Опыт) vs Зарплата', fontsize=14)
plt.xlabel('ln(Опыт)', fontsize=12)
plt.ylabel('Зарплата (тыс. руб)', fontsize=12)
plt.grid(True, alpha=0.3)

# Добавляем линию регрессии на преобразованных данных
X_log_flat = X_log.flatten()
z = np.polyfit(X_log_flat, y, 1)
p = np.poly1d(z)
plt.plot(np.sort(X_log_flat), p(np.sort(X_log_flat)), 'red', linewidth=2, label='Линейная регрессия')
plt.legend()
plt.show()